# Vector intersection lab

Extract one page's raw vectors and cluster them exactly as P3 **VectorClassification** does. The chain is
layer -> color -> stroke-width buckets -> seqno-overlap merge -> constrained spatial clustering, via
`classify_vectors`.

Then, **within each cluster**, count for every `Vector` (one whole `get_drawings()` drawing) how many *other*
Vectors of that cluster it **properly crosses**:
- Only interior X-crossings count.
- Touching endpoints, corners and T-junctions don't count.
- A Vector crossing its own items doesn't count.
- Curves are flattened before the test.

Colour ramp: **0 = red, 3 = yellow, 5 = green, 7+ = blue**, linear in between.

Writes a `scripts/pipeline_report_viewer.py`-compatible folder under `outputs/vector_intersection_lab/`, plus a
histogram. Helpers live in `rastervec/notebooks/_vector_probe_helpers.py`.

## 0 - Config

In [ ]:
PDF_PATH = None     # None -> first PDF under references/
PAGE = 0
CROSS_EPS = 0.01    # pt; an endpoint closer than this to the other segment's line is "touching", not crossing
CURVE_STEPS = 8     # segments per flattened 'c' item
LINE_WIDTH = 1.0    # min overlay stroke width (pt) in the debug PDFs

## 1 - Bootstrap

In [ ]:
import colorsys
import os, sys
from pathlib import Path

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

import matplotlib.pyplot as plt
import numpy as np

from rastervec.commons.helpers.geometry import union_bbox
from rastervec.notebooks import _vector_probe_helpers as h

if PDF_PATH is None:
    refs = sorted((Path(_root) / 'references').glob('*.pdf'))
    assert refs, 'no PDFs found under references/'
    PDF_PATH = str(refs[0])
print(PDF_PATH, 'page', PAGE)

## 2 - Extract + cluster (P3 VectorClassification chain)

In [ ]:
page_meta, vectors = h.load_page_vectors(PDF_PATH, PAGE)
clusters = h.cluster_vectors(vectors, page_meta)
print(f'{len(vectors)} vectors -> {len(clusters)} clusters '
      f'across {len({k for k, _ in clusters})} layer/color/width buckets')

## 3 - Crossing partners per Vector, within each cluster

In [ ]:
crossings = {}   # id(v) -> distinct crossing partners
records = []     # (cluster_index, bucket_key, vector, count)
for ci, (key, members) in enumerate(clusters):
    for v, n in zip(members, h.crossing_partner_counts(members, eps=CROSS_EPS, curve_steps=CURVE_STEPS)):
        crossings[id(v)] = n
        records.append((ci, key, v, n))

all_counts = [n for *_, n in records]
print(f'{len(records)} vectors scored; max crossings = {max(all_counts, default=0)}')

## 4 - Debug layers

In [ ]:
report = h.DebugReport('vector_intersection_lab', PDF_PATH, page_meta, line_width=LINE_WIDTH)
all_vectors = [v for _, _, v, _ in records]

# Low counts first so the busier (bluer) vectors end up on top.
ramp_order = sorted(all_vectors, key=lambda v: crossings[id(v)])
report.add_vectors('intersections', 'all (ramp)', h.ramp_crossings(3), ramp_order,
                   lambda v: h.ramp_crossings(crossings[id(v)]))

for k in range(8):
    label = '7+ crossings' if k == 7 else f'{k} crossing' + ('' if k == 1 else 's')
    members = [v for v in all_vectors if (crossings[id(v)] >= 7 if k == 7 else crossings[id(v)] == k)]
    report.add_vectors('intersections', label, h.ramp_crossings(k), members,
                       lambda v, k=k: h.ramp_crossings(k))

hues = h.golden_hues(len(clusters))
report.add_boxes('clusters', 'cluster bboxes', '#888888', [
    (union_bbox([v.bbox for v in members]), colorsys.hsv_to_rgb(hues[ci], 0.8, 0.8))
    for ci, (_, members) in enumerate(clusters)
])

## 5 - Histogram

In [ ]:
binned = [min(n, 7) for n in all_counts]
heights = [binned.count(k) for k in range(8)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(8), heights, color=[h.ramp_crossings(k) for k in range(8)], edgecolor='black', linewidth=0.5)
ax.set_xticks(range(8), [str(k) for k in range(7)] + ['7+'])
ax.set_xlabel('distinct crossing partners (within cluster)')
ax.set_ylabel('vectors')
ax.set_title(f'Crossings per Vector -- {Path(PDF_PATH).stem} p{PAGE}')
for x, y in zip(range(8), heights):
    ax.text(x, y, str(y), ha='center', va='bottom', fontsize=8)
fig.tight_layout()
report.save_figure(fig, 'histogram_crossings')
plt.show()

## 6 - Per-cluster summary (top 15 by mean crossings, clusters with >= 3 vectors)

In [ ]:
by_cluster = {}
for ci, key, v, n in records:
    by_cluster.setdefault(ci, (key, []))[1].append(n)
rows = sorted(((np.mean(ns), ci, key, ns) for ci, (key, ns) in by_cluster.items() if len(ns) >= 3),
              key=lambda r: -r[0])[:15]
print(f"{'cluster':>7} {'n':>5} {'mean':>6} {'max':>4}  bucket (layer, color, width)")
for mean, ci, key, ns in rows:
    print(f'{ci:>7} {len(ns):>5} {mean:>6.2f} {max(ns):>4}  {key}')

## 7 - Write manifest

In [ ]:
out = report.finish()
print('wrote', out)
for e in report.layers:
    print(f"  {e['stage']:>14} | {e['layer']}")
print()
print('open with:')
print(report.viewer_command())